# 🎬 Storyteller — One-Button YouTube Video Studio

**No computer needed.** Run this notebook in Google Colab on your phone, then open the public link in Chrome.

Colab does all the work (research, writing, images, voice, video rendering). Your phone is just the remote control.

---

### What each cell does
1. **Mount Google Drive** — saves your projects so they survive Colab disconnects
2. **Install Storyteller** — clones the repo and installs all Python packages
3. **Verify real narration** — checks Edge-TTS can speak (needs internet)
4. **Verify real images** — checks Openverse can find photos (needs internet)
5. **Launch the app** — starts the server and gives you a phone-friendly link
6. **YouTube login (optional)** — one-time OAuth so you can publish directly

Run every cell in order. When Cell 5 finishes, copy the `https://...` link and open it on your itel A60 in Chrome.

In [15]:
# Cell 1 — Mount Google Drive (projects saved here, survive disconnects)
from google.colab import drive

print("🔐 Click the link below, sign in to your Google account, and paste the code back.")
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/YouTubeStudio/Projects', exist_ok=True)

print("✅ Drive mounted. Projects will save to:")
print("   /content/drive/MyDrive/YouTubeStudio/Projects/")

🔐 Click the link below, sign in to your Google account, and paste the code back.
Mounted at /content/drive
✅ Drive mounted. Projects will save to:
   /content/drive/MyDrive/YouTubeStudio/Projects/


In [16]:
# Cell 2 — Install Storyteller (run once; re-running is safe)
import os, subprocess

if not os.path.exists('/content/Storyteller'):
    print("⬇️  Downloading Storyteller from GitHub...")
    result = subprocess.run(
        ['git', 'clone', '-b', 'arena/5f37fbb5-storyteller',
         'https://github.com/mohamtur1/Storyteller.git',
         '/content/Storyteller'],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        print("❌ Git clone failed. Make sure the repo is public or check your internet.")
        print(result.stderr[-500:])
    else:
        print("✅ Storyteller downloaded.")
else:
    print("✅ Storyteller already downloaded.")

%cd /content/Storyteller
!pip install -q -r requirements.txt

print("✅ Storyteller installed.")

✅ Storyteller already downloaded.
/content/Storyteller
✅ Storyteller installed.


/usr/lib/python3.13/pathlib/_local.py:274: RuntimeWarning: coroutine '_edge' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [17]:
# Cell 3 — Point projects at Google Drive
import os

if not os.path.exists('/content/Storyteller'):
    print("❌ /content/Storyteller not found. Please run Cell 2 first.")
else:
    os.chdir('/content/Storyteller')
    os.environ["PROJECTS_DIR"] = "/content/drive/MyDrive/YouTubeStudio/Projects"

    from app.config import settings

    print("📁 Projects dir:", settings.projects_dir)
    print("✅ Ready.")

📁 Projects dir: /content/drive/MyDrive/YouTubeStudio/Projects
✅ Ready.


In [18]:
# Cell 4 — Verify REAL narration (Edge-TTS, needs internet)
import asyncio
from pathlib import Path
from app.tts import synthesize_scene

# Patch asyncio to allow running inside Colab's existing event loop
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

test = Path('/tmp/tts_test.mp3')
result = synthesize_scene("Hello. This is a real voice test for Storyteller.", test)

if result["engine"] != "silent":
    print(f"✅ Real narration works — engine: {result['engine']}")
    print(f"   Audio file: {result['path']} ({test.stat().st_size} bytes)")
else:
    print("☀️  Edge-TTS not available (no internet?). Narration will be silent with subtitles.")
    print("   Fix: ensure Colab has internet, then re-run this cell.")

✅ Real narration works — engine: edge-tts
   Audio file: /tmp/tts_test.mp3 (28800 bytes)


In [19]:
# Cell 5 — Verify REAL images (Openverse, needs internet)
from pathlib import Path
from app.images import openverse, _download

cands = openverse("sky blue nature", count=3)
got = None
for c in cands:
    dest = Path('/tmp/img_test.jpg')
    if _download(c["url"], dest):
        got = c
        break

if got:
    print(f"✅ Real images work — found: {got['title']}")
    print(f"   Source: {got['source']}")
    print(f"   License: {got['license']}")
else:
    print("⚠️  Openverse not reachable (no internet?). Images will be gradient slides.")
    print("   Fix: ensure Colab has internet, then re-run this cell.")

✅ Real images work — found: Reach toward the sky. #blue #nature #plants #California #beach #clouds #spring
   Source: https://www.flickr.com/photos/56106041@N00/8636186673
   License: by-nc


In [20]:
# Cell 6 — Launch Storyteller with a public phone-friendly link
import os, subprocess, time

# 1. Start the FastAPI server on port 8000 using python subprocess instead of thread
# (This completely avoids the nest_asyncio / uvicorn runner conflict in the notebook)
print("🚀 Starting Storyteller server...")
server_process = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3) # Wait for server to bind

# 2. Expose the server using localtunnel (bypasses ngrok's authentication requirement)
print("🌐 Setting up public link via localtunnel (no account needed)...")
!npm install -g localtunnel > /dev/null 2>&1

# Launch localtunnel in the background
lt_process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait and read the public URL
public_url = ""
for _ in range(10):
    time.sleep(1)
    # Check if we got a line with the URL
    try:
        line = lt_process.stdout.readline()
        if "your url is" in line:
            public_url = line.split("is:")[-1].strip()
            break
    except Exception:
        pass

if not public_url:
    public_url = "Could not retrieve localtunnel link automatically. Please try re-running this cell."

sep = "━" * 45
print(sep)
print("🎉 STORYTELLER IS LIVE!")
print(sep)
print()
print(f"🔗 Public link: {public_url}")
print()
print("📱 On your itel A60:")
print("   1. Open Chrome")
print(f"   2. Go to: {public_url}")
print("   3. Type a topic → Generate Video → Preview → Publish")
print()
print("💡 Keep this Colab tab open while you make videos.")
print("   If it disconnects, re-run Cells 1–6; your projects are safe on Drive.")
print()
print(sep)

🚀 Starting Storyteller server...
🌐 Setting up public link via localtunnel (no account needed)...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎉 STORYTELLER IS LIVE!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔗 Public link: https://tame-boxes-hammer.loca.lt

📱 On your itel A60:
   1. Open Chrome
   2. Go to: https://tame-boxes-hammer.loca.lt
   3. Type a topic → Generate Video → Preview → Publish

💡 Keep this Colab tab open while you make videos.
   If it disconnects, re-run Cells 1–6; your projects are safe on Drive.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# Cell 7 — YouTube login (optional — skip for manual upload)
# Do this only if you want one-tap publishing to YouTube.
# It requires a one-time Google account authorization.

!python -m app.youtube_auth

---

## 📱 Using Storyteller on your itel A60

1. Copy the public link from Cell 6
2. Open **Chrome** on your phone
3. Paste the link and press Go
4. Type a topic (e.g. *Why is the sky blue?*)
5. Tap **✨ Generate Video**
6. Watch the progress — Colab is rendering the video
7. When it says **Ready to review**, watch the video and read the subtitles
8. Tap **🚀 Publish to YouTube** (or save the MP4 to Drive)

### Where your files go

Every project is saved as a folder in:
`MyDrive/YouTubeStudio/Projects/<ProjectName>/`

Inside each project folder:
- `FinalVideo/video.mp4` — your finished video
- `Thumbnail/thumbnail.jpg` — the cover image
- `Subtitles/captions.srt` — the subtitles
- `Script/script.txt` — the full script
- `Metadata/description.txt` — title, description, tags
- `Metadata/manual_upload/` — ready-to-upload package if YouTube API isn't set up

### YouTube publishing

The video is **unlisted by default** — only you (and anyone with the link) can see it.
You must tap **Publish** yourself; nothing is uploaded without your explicit approval.

If you didn't set up YouTube OAuth (Cell 7), the app gives you a **manual-upload package**:
download the MP4 from Drive, then upload it at youtube.com/upload in 2 minutes.

### If Colab disconnects

Your projects are safe on Drive. Come back, re-run Cells 1–6, and the app picks up where it left off.

---

## ⚠️ What needs your Google account

| Step | What happens | Why your account is needed |
|------|-------------|---------------------------|
| Cell 1 | Mount Google Drive | So your videos are saved to *your* Drive, not lost when Colab ends |
| Cell 7 | YouTube OAuth | So Storyteller can upload to *your* YouTube channel (one-time setup) |

Everything else works with zero accounts and zero API keys.